In [1]:
!pip install pandas matplotlib numpy requests python-dotenv tidalapi

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from spotify_api import get_access_token as get_spotify_access_token
from tidal_api import create_session as create_tidal_session

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
SPOTIFY_CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')
if not SPOTIFY_CLIENT_ID or not SPOTIFY_CLIENT_SECRET:
    raise ValueError("Spotify client ID and secret must be set in environment variables.")

SPOT_HEADERS = get_spotify_access_token(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
print("Authenticated with Spotify successfully.")

TIDAL_SESSION = create_tidal_session(Path("tidal_session.json"))
print("Authenticated with TIDAL successfully.")

/home/shearer/personal/playlist_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated with Spotify successfully.
Authenticated with TIDAL successfully.


In [3]:
from spotify_api import get_user_access_token, get_user_top_artists

SPOTIFY_USER_HEADERS = get_user_access_token(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
print("Authenticated with Spotify OAuth successfully.")

top_artists = get_user_top_artists(SPOTIFY_USER_HEADERS, time_range="long_term")
ARTIST_NAMES = [a["name"] for a in top_artists]
print(f"\n{len(ARTIST_NAMES)} artists:")
    
ARTIST_NAMES = ARTIST_NAMES[:10]
for name in ARTIST_NAMES:
    print(f"  - {name}")

Opening browser for Spotify login...
https://accounts.spotify.com/authorize?client_id=77f4c3abb6ab43b2aec536d8f9adf301&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A5000%2Fcallback%2Fspotify&scope=user-top-read+user-read-recently-played
Authenticated with Spotify OAuth successfully.
Found 223 top artists (long_term)

223 artists:
  - Søte & Rare
  - Ari Bajgora
  - Cameron Whitcomb
  - Luther
  - Fjellrev
  - Ninho
  - Roc Boyz
  - L-Story
  - GOLF
  - Problembarn


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from spotify_api import get_artist_song_ids
from tidal_api import spotify_to_tidal_ids

# Fetch song IDs for all artists in parallel
spotify_song_ids: list[str] = []
with ThreadPoolExecutor(max_workers=1) as pool:
    futures: dict = {pool.submit(get_artist_song_ids, name, SPOT_HEADERS): name for name in ARTIST_NAMES}
    for future in as_completed(futures):
        try:
            spotify_song_ids.extend(future.result())
        except Exception as e:
            print(f"[ERROR] {futures[future]}: {e}")

print(f"{len(spotify_song_ids)} Spotify song IDs")
tidal_song_ids: list[dict] = spotify_to_tidal_ids(spotify_song_ids, SPOT_HEADERS, TIDAL_SESSION)
print(tidal_song_ids[:10])

[Søte & Rare] Searching...


DEBUG:urllib3.util.retry:Incremented Retry for (url='/v1/artists/2gmup32OWc65WVKSoP4rD8/albums?include_groups=album%2Csingle%2Ccompilation&limit=50'): _LoggingRetry(total=4, connect=None, read=None, redirect=None, status=None)


[Søte & Rare] Found artist (ID: 2gmup32OWc65WVKSoP4rD8)
[Søte & Rare] Fetching albums page 0...
  [429] Rate-limited — waiting 21600s before retry...


In [ ]:
from db_connection import db, Songs, SongAudioFeatures

db.connect(reuse_if_open=True)
db.create_tables([Songs, SongAudioFeatures], safe=True)
print("Tables ready.")
db.close()

Tables ready.


True

In [ ]:
db.connect(reuse_if_open=True)
existing_spotify_ids = set(s.spotify_id for s in Songs.select(Songs.spotify_id))
db.close()

before = len(spotify_song_ids)
spotify_song_ids = [sid for sid in spotify_song_ids if sid not in existing_spotify_ids]
print(f"Filtered {before - len(spotify_song_ids)} existing songs, {len(spotify_song_ids)} remaining")

# Re-derive TIDAL IDs for the filtered list
tidal_song_ids = spotify_to_tidal_ids(spotify_song_ids, SPOT_HEADERS, TIDAL_SESSION)
print(f"Mapped {len(tidal_song_ids)} songs to TIDAL IDs")

Filtered 144 existing songs, 4 remaining


Looking up TIDAL IDs: 100%|██████████| 4/4 [00:00<00:00, 11.65it/s]

Mapped 4 songs to TIDAL IDs


In [ ]:
import uuid
from spotify_api import get_tracks_metadata

tracks_meta = get_tracks_metadata(spotify_song_ids, SPOT_HEADERS)
print(f"Fetched metadata for {len(tracks_meta)} tracks")

# Build a lookup from spotify_id -> tidal_id
tidal_lookup = {entry["spotify_id"]: entry["tidal_id"] for entry in tidal_song_ids}

db.connect(reuse_if_open=True)
inserted, skipped = 0, 0
for meta in tracks_meta:
    try:
        Songs.create(
            id=uuid.uuid4(),
            spotify_id=meta["id"],
            tidal_id=tidal_lookup.get(meta["id"]),
            song_name=meta["name"],
            artists=meta["artists"],
            album_name=meta.get("album_name"),
            album_total_tracks=meta.get("album_total_tracks"),
            release_date=meta.get("album_release_date"),
            disc_number=meta.get("disc_number"),
            duration_ms=meta.get("duration_ms"),
            explicit=meta.get("explicit"),
            isrc=meta.get("isrc"),
            spotify_external_url=meta.get("external_url"),
            spotify_href=meta.get("href"),
            popularity=meta.get("popularity"),
            track_number=meta.get("track_number"),
            spotify_uri=meta.get("uri"),
        )
        inserted += 1
    except Exception as e:
        skipped += 1  # likely duplicate song_name or spotify_id

db.close()
print(f"Inserted {inserted} songs, skipped {skipped} duplicates")

Fetched metadata for 4 tracks
Inserted 0 songs, skipped 4 duplicates


In [ ]:
from tidal_api import download_songs

downloaded_paths = download_songs(tidal_song_ids)
print(f"Downloaded {len(downloaded_paths)} tracks")

Downloaded 4 tracks


In [ ]:
from pathlib import Path
from audio_processor import process_bulk

# Gather downloaded audio files keyed by tidal_id
downloads_dir = Path("OrpheusDL/downloads")
audio_files = {}
for entry in tidal_song_ids:
    tid = entry["tidal_id"]
    if tid is None:
        continue
    path = downloads_dir / f"{tid}.m4a"
    if path.exists():
        audio_files[tid] = path

print(f"Found {len(audio_files)} audio files to analyse")

# Bulk analyse
paths_list = list(audio_files.values())
tidal_ids_list = list(audio_files.keys())
analysis_results = process_bulk(paths_list)

# Store in DB
db.connect(reuse_if_open=True)
stored, failed = 0, 0
for tid, analysis in zip(tidal_ids_list, analysis_results):
    if "error" in analysis:
        print(f"[SKIP] {tid}: {analysis['error']}")
        failed += 1
        continue

    song = Songs.get_or_none(Songs.tidal_id == tid)
    if song is None:
        failed += 1
        continue

    track_info = analysis.get("track", {})
    try:
        SongAudioFeatures.create(
            song=song.id,
            acousticness=analysis.get("acousticness"),
            danceability=analysis.get("danceability"),
            energy=analysis.get("energy"),
            instrumentalness=analysis.get("instrumentalness"),
            key=analysis.get("key"),
            liveness=analysis.get("liveness"),
            loudness=analysis.get("loudness"),
            mode=analysis.get("mode"),
            speechiness=analysis.get("speechiness"),
            tempo=analysis.get("tempo"),
            time_signature=analysis.get("time_signature"),
            valence=analysis.get("valence"),
            num_samples=track_info.get("num_samples"),
            duration=track_info.get("duration"),
            sample_md5=track_info.get("sample_md5"),
            end_of_fade_in=track_info.get("end_of_fade_in"),
            start_of_fade_out=track_info.get("start_of_fade_out"),
            tempo_confidence=track_info.get("tempo_confidence"),
            time_signature_confidence=track_info.get("time_signature_confidence"),
            key_confidence=track_info.get("key_confidence"),
            mode_confidence=track_info.get("mode_confidence"),
            bars=analysis.get("bars"),
            beats=analysis.get("beats"),
            sections=analysis.get("sections"),
            segments=analysis.get("segments"),
            tatums=analysis.get("tatums"),
        )
        stored += 1
    except Exception as e:
        print(f"[SKIP] {tid}: {e}")
        failed += 1

db.close()
print(f"Stored audio features for {stored} tracks, failed {failed}")

NameError: name 'tidal_song_ids' is not defined

In [ ]:
from tidal_api import get_track_lyrics
from tqdm.auto import tqdm

db.connect(reuse_if_open=True)
songs_needing_lyrics = (
    Songs.select()
    .where(Songs.tidal_id.is_null(False) & (Songs.lyrics.is_null(True) | (Songs.lyrics == "")))
)
songs_list = list(songs_needing_lyrics)
print(f"Songs needing lyrics: {len(songs_list)}")

fetched, missed = 0, 0
for song in tqdm(songs_list, desc="Fetching lyrics"):
    lyrics = get_track_lyrics(song.tidal_id, TIDAL_SESSION)
    if lyrics:
        Songs.update(lyrics=lyrics).where(Songs.id == song.id).execute()
        fetched += 1
    else:
        missed += 1

db.close()
print(f"Fetched lyrics for {fetched} songs, {missed} not available")

Songs needing lyrics: 48


Fetching lyrics: 100%|██████████| 48/48 [00:06<00:00,  7.45it/s]

Fetched lyrics for 12 songs, 36 not available


In [ ]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tidal_api import download_song

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

TRANSCRIPTION_PROMPT = """This is a song recording. Transcribe all lyrics exactly as heard, following these rules:
- Preserve all repetitions, including choruses sung multiple times
- If lyrics switch language (e.g. Spanish, French), transcribe them in their original language
- Distinguish spoken/ad-libbed lines with parentheses, e.g. (spoken: yeah, that's right)
- If a word or phrase is genuinely unclear, write [unclear] rather than guessing
- Use one blank line between song sections; do not add section labels
- Transcribe informal or slurred speech phonetically as heard, not corrected to standard spelling
- Include background vocals in brackets, e.g. [background: oh yeah]"""

db.connect(reuse_if_open=True)
still_missing = (
    Songs.select()
    .where(Songs.tidal_id.is_null(False) & (Songs.lyrics.is_null(True) | (Songs.lyrics == "")))
)
songs_to_transcribe = [
    {"id": str(s.id), "tidal_id": s.tidal_id}
    for s in still_missing
]
for i in songs_to_transcribe:
    if not (downloads_dir / f"{i['tidal_id']}.m4a").exists():
        try:
            download_song(i["tidal_id"])
        except Exception as e:
            print(f"[ERROR] Failed to download {i['tidal_id']}: {e}")
            del songs_to_transcribe[songs_to_transcribe.index(i)]
print(f"Songs to transcribe via audio: {len(songs_to_transcribe)}")


def _transcribe_one(song):
    audio_file = downloads_dir / f"{song['tidal_id']}.m4a"
    result = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
        response_format="text",
        prompt=TRANSCRIPTION_PROMPT,
    )
    lyrics = result.strip()
    Songs.update(lyrics=lyrics).where(Songs.id == song["id"]).execute()
    return {"id": song["id"], "tidal_id": song["tidal_id"], "lyrics": lyrics}


transcribed = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(_transcribe_one, s): s for s in songs_to_transcribe}
    for future in as_completed(futures):
        song = futures[future]
        try:
            out = future.result()
            transcribed.append(out)
        except Exception as e:
            print(f"[ERROR] {song['tidal_id']}: {e}")

db.close()
print(f"Transcribed lyrics for {len(transcribed)} songs")

[ERROR] Failed to download 4005079: Download Failed for 4005079
[ERROR] Failed to download 37061723: Download Failed for 37061723
[ERROR] Failed to download 34940903: Download Failed for 34940903
[ERROR] Failed to download 45771071: Download Failed for 45771071
[ERROR] Failed to download 35452621: Download Failed for 35452621
[ERROR] Failed to download 45771076: Download Failed for 45771076
[ERROR] Failed to download 45771079: Download Failed for 45771079
[ERROR] Failed to download 35452620: Download Failed for 35452620
[ERROR] Failed to download 36208138: Download Failed for 36208138
[ERROR] Failed to download 4005084: Download Failed for 4005084
[ERROR] Failed to download 45771085: Download Failed for 45771085
[ERROR] Failed to download 45771075: Download Failed for 45771075
Songs to transcribe via audio: 14
[ERROR] 45771078: [Errno 2] No such file or directory: 'OrpheusDL/downloads/45771078.m4a'
[ERROR] 4005088: [Errno 2] No such file or directory: 'OrpheusDL/downloads/4005088.m4a'


In [ ]:
from embedding_model import create_text_embeddings
import torch

db.connect(reuse_if_open=True)
songs_with_lyrics = (
    Songs.select(Songs, SongAudioFeatures)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.lyrics.is_null(False)
        & (Songs.lyrics != "")
        & SongAudioFeatures.lyrics_embeddings.is_null(True)
    )
)
songs_list = list(songs_with_lyrics)
print(f"Songs needing lyrics embeddings: {len(songs_list)}")

embedded = 0
for song in tqdm(songs_list, desc="Lyrics embeddings"):
    lines = [line.strip() for line in song.lyrics.splitlines() if line.strip()]
    if not lines:
        continue
    line_embeddings = create_text_embeddings(lines)           # (n_lines, 768)
    song_embedding = line_embeddings.mean(dim=0)              # (768,)
    song_embedding = song_embedding / song_embedding.norm()   # re-normalise
    (SongAudioFeatures
     .update(lyrics_embeddings=song_embedding.cpu().tolist())
     .where(SongAudioFeatures.song == song.id)
     .execute())
    embedded += 1

db.close()
print(f"Stored lyrics embeddings for {embedded} songs")

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 1629.25it/s, Materializing param=vision_model.post_layernorm.weight]                      


Songs needing lyrics embeddings: 0


Lyrics embeddings: 0it [00:00, ?it/s]

Stored lyrics embeddings for 0 songs


In [ ]:
from file_metadata_processor import extract_image_from_m4a_mutagen
from embedding_model import create_images_embeddings

db.connect(reuse_if_open=True)
songs_needing_images = (
    Songs.select(Songs, SongAudioFeatures)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.tidal_id.is_null(False)
        & SongAudioFeatures.image_embeddings.is_null(True)
    )
)
songs_list = list(songs_needing_images)
print(f"Songs needing image embeddings: {len(songs_list)}")

BATCH_SIZE = 32
embedded = 0
batch_songs = []
batch_images = []

for song in tqdm(songs_list, desc="Extracting album art"):
    audio_path = downloads_dir / f"{song.tidal_id}.m4a"
    if not audio_path.exists():
        continue
    img = extract_image_from_m4a_mutagen(str(audio_path))
    if img is None:
        continue
    batch_songs.append(song)
    batch_images.append(img)

    if len(batch_images) >= BATCH_SIZE:
        embeddings = create_images_embeddings(batch_images)
        embeddings_list = embeddings.cpu().tolist()
        for s, emb in zip(batch_songs, embeddings_list):
            (SongAudioFeatures
             .update(image_embeddings=emb)
             .where(SongAudioFeatures.song == s.id)
             .execute())
            embedded += 1
        batch_songs, batch_images = [], []

# Process remaining batch
if batch_images:
    embeddings = create_images_embeddings(batch_images)
    embeddings_list = embeddings.cpu().tolist()
    for s, emb in zip(batch_songs, embeddings_list):
        (SongAudioFeatures
         .update(image_embeddings=emb)
         .where(SongAudioFeatures.song == s.id)
         .execute())
        embedded += 1

db.close()
print(f"Stored image embeddings for {embedded} songs")

Songs needing image embeddings: 0


Extracting album art: 0it [00:00, ?it/s]

Stored image embeddings for 0 songs


In [ ]:
from db_connection import db, Songs, SongAudioFeatures

db.connect(reuse_if_open=True)

# Songs with audio features but missing lyrics or embeddings
incomplete = (
    Songs.select(Songs.id)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.lyrics.is_null(True)
        | (Songs.lyrics == "")
        | SongAudioFeatures.lyrics_embeddings.is_null(True)
        | SongAudioFeatures.image_embeddings.is_null(True)
    )
)
# Songs with no audio features at all
no_features = (
    Songs.select(Songs.id)
    .left_outer_join(SongAudioFeatures)
    .where(SongAudioFeatures.song.is_null(True))
)
incomplete_ids = [s.id for s in incomplete] + [s.id for s in no_features]

if incomplete_ids:
    feat_del = SongAudioFeatures.delete().where(SongAudioFeatures.song.in_(incomplete_ids)).execute()
    song_del = Songs.delete().where(Songs.id.in_(incomplete_ids)).execute()
    print(f"Deleted {song_del} songs and {feat_del} audio feature rows")
else:
    print("No incomplete songs to delete")

db.close()

Deleted 76 songs and 0 audio feature rows


True